# 1 Select Final Columns
Prune the dataset to a clean final output with only the essential columns.

Input:  data/1_derived/5_geocode_truck_stops/4_final_coordinates.csv
Output: data/1_derived/5_geocode_truck_stops/7_final_truck_stops.csv

In [1]:
from pathlib import Path
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE  = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '4_final_coordinates.csv'
OUT_DIR  = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / '7_final_truck_stops.csv'

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')

Loaded 38,135 rows, 143 columns


In [2]:
# Select final columns
columns_to_keep = [
    'clean_line1',
    'clean_line2',
    'line3',
    'city',
    'zip_code',
    'label',
    'phone',
    'year',
    'major_city',
    'state',
    'chain',
    'Address_Type',
    'Exit_Number',
    'Exit_From_Address',
    'Exit_From_Label',
    'Main_Road',
    'Secondary_Road',
    'Exit_Number_2',
    'Exit_Number_3',
    'Tertiary_Road',
    'Scraped_phone_match_rate',
    'Yelp_phone_match_rate',
    'place_identifier(year)',
    'Yellowbook_phone_match_rate',
    'Webscraped_Phone_full_url',
    'Webscraped_PlacedMatched_full_url',
    'Yelp_URL',
    'YellowPages_SEARCH_URL',
    'Match_Comments',
    'Final_Lat',
    'Final_Long',
]

existing = [c for c in columns_to_keep if c in df.columns]
missing = [c for c in columns_to_keep if c not in df.columns]
df = df[existing]

print(f'Columns kept   : {len(existing)}')
if missing:
    print(f'Columns missing: {missing}')
print(f'Final shape    : {df.shape}')

Columns kept   : 30
Columns missing: ['Webscraped_PlacedMatched_full_url']
Final shape    : (38135, 30)


In [3]:
# Save
df.to_csv(OUT_FILE, index=False)
print(f'Saved: {OUT_FILE}')

Saved: C:\Users\Owner\Desktop\Geocoding_Truck_Stops\data\1_derived\5_geocode_truck_stops\7_final_truck_stops.csv


In [4]:
# Completion-rate check
total = len(df)
has_lat  = df['Final_Lat'].notna()
has_long = df['Final_Long'].notna()
has_both = has_lat & has_long

print(f'Total rows          : {total:,}')
print(f'Has Final_Lat       : {has_lat.sum():,}  ({has_lat.sum()/total*100:.2f}%)')
print(f'Has Final_Long      : {has_long.sum():,}  ({has_long.sum()/total*100:.2f}%)')
print(f'Has BOTH (complete) : {has_both.sum():,}  ({has_both.sum()/total*100:.2f}%)')
print(f'Missing either      : {(~has_both).sum():,}  ({(~has_both).sum()/total*100:.2f}%)')

# Breakdown
print(f'\n-- Rows with lat but no long (or vice-versa) --')
print(f'  Lat only : {(has_lat & ~has_long).sum():,}')
print(f'  Long only: {(~has_lat & has_long).sum():,}')
print(f'  Both null: {(~has_lat & ~has_long).sum():,}')

if 'year' in df.columns:
    print(f'\n-- Completion rate by year --')
    by_year = df.groupby('year').apply(
        lambda g: pd.Series({
            'total': len(g),
            'complete': (g['Final_Lat'].notna() & g['Final_Long'].notna()).sum()
        })
    )
    by_year['pct'] = (by_year['complete'] / by_year['total'] * 100).round(2)
    print(by_year.to_string())

Total rows          : 38,135
Has Final_Lat       : 33,473  (87.78%)
Has Final_Long      : 33,473  (87.78%)
Has BOTH (complete) : 33,473  (87.78%)
Missing either      : 4,662  (12.22%)

-- Rows with lat but no long (or vice-versa) --
  Lat only : 0
  Long only: 0
  Both null: 4,662

-- Completion rate by year --
      total  complete    pct
year                        
2006   6480      5154  79.54
2007   6385      5231  81.93
2008   6224      5246  84.29
2014   6372      5889  92.42
2015   6339      5936  93.64
2016   6335      6017  94.98
